# Spectral Folding for ODMR Data

**Spectral folding** exploits the mirror symmetry of ODMR spectra about the zero-field splitting D:

```
f+/- = D +/- gamma * B * cos(theta)
```

The low and high frequency ranges are mirror images about D ~ 2.870 GHz. Averaging them gives:

- **sqrt(2) SNR improvement** in the combined spectrum
- **D_ZFS map** per pixel (temperature/strain proxy)
- **Fold residual** -- a model-free data quality metric

This notebook has two parts:

| Part | Audience | Style |
|------|----------|-------|
| **A. Quick path** | Fry -- "just give me B111 maps" | 3 API calls, done |
| **B. Detailed path** | Lila -- "I need to tweak settings" | Manual SpectralFolder, diagnostics, custom fitting |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qdmpy.measurement import Measurement
from qdmpy.constants import D_ZFS, D_ZFS_TEMP_COEFFICIENT
from qdmpy.odmr import FoldingSettings

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

## Load data

We load the MIL2_FOV1 dataset, 4x4 spatially binned for speed.
Both Part A and Part B use the same `Measurement` object -- no extra RAM.

In [ ]:
m = Measurement.from_folder('/home/mike/Documents/FOV18x', bin_factor=1, normalize=True)
data = m.odmr.processed_data.data
freq_ghz = data.coords['freq_ghz'].values   # (n_frange, n_freq)

print(f'Shape:      {data.shape}  (pol, frange, y, x, freq)')
print(f'Freq low:   {freq_ghz[0, 0]:.4f} - {freq_ghz[0, -1]:.4f} GHz')
print(f'Freq high:  {freq_ghz[1, 0]:.4f} - {freq_ghz[1, -1]:.4f} GHz')

In [ ]:
# Inspect a single-pixel ODMR spectrum (low + high ranges)
cy, cx = data.sizes['y'] // 2, data.sizes['x'] // 2

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(freq_ghz[0], data.sel(polarity='neg', freq_range='low').values[cy, cx],
        'tab:blue', lw=1.5, label='low range (f-)')
ax.plot(freq_ghz[1], data.sel(polarity='neg', freq_range='high').values[cy, cx],
        'tab:orange', lw=1.5, label='high range (f+)')
ax.axvline(D_ZFS, color='0.4', ls='--', lw=1, label=f'D_ZFS = {D_ZFS} GHz')
ax.set(xlabel='Frequency (GHz)', ylabel='Normalised intensity',
       title=f'ODMR spectrum at pixel ({cy}, {cx}), pol=neg')
ax.legend()
plt.tight_layout()
plt.show()

The two dips are roughly symmetric about D_ZFS. Folding them together about the
correct D gives a single dip on a Zeeman-offset axis `delta_f` with sqrt(2)
better SNR.

---

# Part A: Quick path (Fry)

Three calls: `fold_odmr()` -> diagnostic check -> `fit_folded_odmr()`.

## A1. Fold

`m.fold_odmr()` runs the full two-scale folding pipeline and caches the result.
Default settings work well for most datasets.

In [ ]:
folded = m.fold_odmr(settings=FoldingSettings(search_range=0.003))

print(f'Folded spectrum: shape={tuple(folded.folded_spectrum.shape)}')
print(f'D_ZFS map:       shape={tuple(folded.d_zfs_map.shape)}')
print(f'Fold residual:   shape={tuple(folded.fold_residual.shape)}')

## A2. Diagnostic check

`folded.plot()` gives a 2x2 overview: search landscape (did the brute-force
find a clean minimum?), D_ZFS deviation map, and fold residual map.

In [ ]:
folded.plot()

## A3. Fit and extract B111

`fit_folded_odmr()` uses the cached folded data. The folded spectrum's frequency
axis is already a Zeeman offset, so the fitted centre IS `delta_f` directly --
no D_ZFS subtraction needed for B111 maps.

In [ ]:
qdm_result = m.fit_folded_odmr()

print(f'B111 remanent: mean={qdm_result.b111_remanent.mean():.1f} uT, '
      f'std={qdm_result.b111_remanent.std():.1f} uT')
print(f'B111 induced:  mean={qdm_result.b111_induced.mean():.1f} uT, '
      f'std={qdm_result.b111_induced.std():.1f} uT')

In [ ]:
qdm_result.get_fit_quality_metrics()

In [ ]:
from qdmpy.plotting import plot_b111_map

# Prefer predefined plotting helpers when available
plot_b111_map(qdm_result.fit_result, component='remanent')
plot_b111_map(qdm_result.fit_result, component='induced')


In [ ]:
from qdmpy.plotting import plot_folding_pixel_spectra
plot_folding_pixel_spectra(folded, x=0, y=[0,10,140])

## B1. Configure and fold

The key tuning knobs:

| Setting | Default | Effect |
|---------|---------|--------|
| `bin_factor` | 8 | Spatial binning for coarse D search (higher = faster, coarser) |
| `search_steps` | 201 | D candidates over +/-`search_range` (more = finer D resolution) |
| `search_range` | 0.005 GHz | Half-width of D search (+/-5 MHz covers ~135 K range) |
| `min_overlap_points` | 5 | Minimum valid delta_f points per candidate |

In [ ]:
from qdmpy.odmr.folding import SpectralFolder

settings = FoldingSettings(
    bin_factor=1,
    search_steps=201,
    search_range=0.003,
)

# Uses the same processed data -- no extra RAM
folder = SpectralFolder(m.odmr.processed_data, settings)
result = folder.fold()

print(f'Folded spectrum: {tuple(result.folded_spectrum.shape)}')
print(f'D_ZFS map:       {tuple(result.d_zfs_map.shape)}')
print(f'delta_f axis:    {result.folded_spectrum.coords["delta_f_ghz"].values[0]*1000:.2f} - '
      f'{result.folded_spectrum.coords["delta_f_ghz"].values[-1]*1000:.2f} MHz '
      f'({result.folded_spectrum.sizes["freq_idx"]} points)')

## B2. Search landscape diagnostic

The brute-force search sweeps D candidates and picks the one that minimises the
low-vs-high mismatch. A clean, single minimum means the search is well-behaved.
Multiple minima or a flat landscape suggests the search range or frequency
coverage may need adjustment.

In [ ]:
from qdmpy.plotting import plot_folding_search_landscape
plot_folding_search_landscape(result)

## B3. Mean folded spectrum

The spatially-averaged folded spectrum should show a clear ODMR dip centred at
the Zeeman offset. The antisymmetric component (red) should be near zero --
large antisymmetric residuals indicate D_ZFS estimation errors or asymmetric
line shapes.

In [ ]:
from qdmpy.plotting import plot_folding_mean_spectrum
plot_folding_mean_spectrum(result)

## B4. D_ZFS and temperature maps

The per-pixel D_ZFS map doubles as a temperature/strain proxy:
dD/dT = -74 kHz/K, so 1 MHz deviation ~ 13.5 K.

In [ ]:
from qdmpy.plotting import plot_folding_overview

# Use the built-in folding diagnostics overview (search + dD + residual)
plot_folding_overview(result)

# Temperature variation map (no dedicated helper yet)
pixel_um = m.pixel_spacing * 1e6
d_zfs_neg = result.d_zfs_map.sel(polarity='neg').values
delta_T = (d_zfs_neg - D_ZFS) / D_ZFS_TEMP_COEFFICIENT

ny, nx = d_zfs_neg.shape
ext = [0, nx * pixel_um, ny * pixel_um, 0]

fig, ax = plt.subplots(1, 1, figsize=(6, 4))
im = ax.imshow(delta_T, extent=ext, cmap='RdBu_r')
ax.set_title('Temperature variation dT (K)')
ax.set_xlabel('x (um)')
ax.set_ylabel('y (um)')
fig.colorbar(im, ax=ax, label='dT (K)')

plt.tight_layout()
plt.show()

print(f'D_ZFS range: {(d_zfs_neg.min()-D_ZFS)*1000:.3f} - {(d_zfs_neg.max()-D_ZFS)*1000:.3f} MHz')
print(f'dT range:    {delta_T.min():.1f} - {delta_T.max():.1f} K')


## B5. Fit the folded spectrum

Pass the manually-created `FoldedODMR` result to `fit_folded_odmr()` via the
`folded=` argument. This lets you use custom settings without overwriting the
Measurement's cached result.

In [ ]:
qdm_result_b = m.fit_folded_odmr(folded=result)

print(f'B111 remanent: mean={qdm_result_b.b111_remanent.mean():.1f} uT, '
      f'std={qdm_result_b.b111_remanent.std():.1f} uT')
print(f'B111 induced:  mean={qdm_result_b.b111_induced.mean():.1f} uT, '
      f'std={qdm_result_b.b111_induced.std():.1f} uT')

## Summary

| Step | Quick (Fry) | Detailed (Lila) |
|------|-------------|----------------|
| Fold | `m.fold_odmr()` | `SpectralFolder(m.odmr.processed_data, settings).fold()` |
| Diagnose | `folded.plot()` | `plot_folding_search_landscape()`, `plot_folding_mean_spectrum()` |
| Fit | `m.fit_folded_odmr()` | `m.fit_folded_odmr(folded=result)` |
| D_ZFS map | `m.folded_odmr.d_zfs_map` | `result.d_zfs_map` |
| B111 | `qdm_result.b111_remanent` | same |

The D_ZFS map also enables temperature/strain mapping:
dD/dT = -74 kHz/K, so 1 MHz of dD ~ 13.5 K.

Folded fitting now returns the standard `FitResult`, with centers converted
back to absolute GHz internally; B111 is computed via `(center - D_ZFS)/gamma`
through the same path as non-folded fitting.

In [ ]:
m.display(qdm_result_b)